# Lower precision, measured

float16 inference, mixed-precision training, and the loss scaling without which small gradients round to zero.

**Runs on:** GPU recommended — about 10 minutes &nbsp;·&nbsp; **Slides:** [Chapter 18 — Best Practices for the Real World](../../../course-web-slides/ch18/index.html) &nbsp;·&nbsp; **Section:** 04 — Lower-precision computation

---

## What precision actually means

In [ ]:
import numpy as np

for dtype, name in [(np.float16, "float16"), (np.float32, "float32"),
                    (np.float64, "float64")]:
    info = np.finfo(dtype)
    print(f"{name:9s} bits {info.bits:2d}   eps {info.eps:.2e}   "
          f"max {info.max:.2e}   tiny {info.tiny:.2e}")

**`eps` is the resolution**: the smallest distance between two representable numbers near 1. float16 gives about 1e-3, float32 about 1e-7, float64 about 1e-16.

Typical learning rates are 1e-3 and typical weight updates around 1e-6. **float16 cannot represent that update at all.**

## Representable numbers are not evenly spaced

In [ ]:
import matplotlib.pyplot as plt

def spacing(dtype, values):
    return [float(np.spacing(dtype(v))) for v in values]

vals = np.logspace(-3, 4, 40)
plt.figure(figsize=(7, 4.2))
plt.loglog(vals, spacing(np.float16, vals), "o-", ms=3, label="float16")
plt.loglog(vals, spacing(np.float32, vals), "s-", ms=3, label="float32")
plt.xlabel("magnitude"); plt.ylabel("gap to the next representable number")
plt.legend(); plt.title("Larger numbers have lower precision")
plt.show()

There are as many representable values between 2^N and 2^(N+1) as between 1 and 2, for any N. **The error of converting a number to floating point grows with its magnitude** — which is why normalizing your inputs was never only about gradients.

## float16 or bfloat16

In [ ]:
print(f"{'':16s} {'exponent':>9s} {'mantissa':>9s} {'sign':>5s}")
print(f"{'float16':16s} {5:>9d} {10:>9d} {1:>5d}")
print(f"{'bfloat16':16s} {8:>9d} {7:>9d} {1:>5d}")
print(f"{'float32':16s} {8:>9d} {23:>9d} {1:>5d}")
print()
print("bfloat16 has float32's RANGE with far less resolution.")
print("Some devices -- TPUs especially -- are better optimized for it.")
print("It is a one-line experiment: try both, keep the faster.")

## float16 inference

In [ ]:
import keras
from keras import layers
import time

keras.config.set_dtype_policy("float32")
(x, y), (xt, yt) = keras.datasets.mnist.load_data()
x = x.reshape(-1, 784).astype("float32") / 255
xt = xt.reshape(-1, 784).astype("float32") / 255

def build():
    keras.utils.set_random_seed(0)
    m = keras.Sequential([layers.Dense(512, activation="relu"),
                          layers.Dense(512, activation="relu"),
                          layers.Dense(10, activation="softmax")])
    m.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

fp32 = build()
fp32.fit(x, y, epochs=3, batch_size=128, verbose=0)

def bench(m, data, n=10, warmup=3):
    for _ in range(warmup):
        m.predict(data, verbose=0)
    t0 = time.perf_counter()
    for _ in range(n):
        m.predict(data, verbose=0)
    return (time.perf_counter() - t0) / n

batch = xt[:2048]
t32 = bench(fp32, batch)
print(f"float32: {t32*1000:6.1f} ms   acc {fp32.evaluate(xt, yt, verbose=0)[1]:.4f}")

> ⚠️ **Warm up before timing.** The first call compiles; timing it measures the compiler, not the model.

## Mixed precision for training

In [ ]:
keras.config.set_dtype_policy("mixed_float16")

mixed = build()
t0 = time.time()
mixed.fit(x, y, epochs=3, batch_size=128, verbose=0)
t_mixed = time.time() - t0

print(f"mixed_float16 training: {t_mixed:.1f}s   "
      f"acc {mixed.evaluate(xt, yt, verbose=0)[1]:.4f}")
print()
for layer in mixed.layers:
    print(f"{layer.name:12s} compute {layer.compute_dtype:12s} "
          f"variables {layer.variable_dtype}")

**`compute_dtype` is float16; `variable_dtype` stays float32.** Most of the forward pass runs on half-precision copies of the weights; the weights themselves are stored and updated in full precision, so they can receive accurate small updates.

Some operations are numerically unstable in float16 — notably softmax and crossentropy. Pass `dtype="float32"` to opt a specific layer out.

## Loss scaling

In [ ]:
import numpy as np

# Gradients that vanish in float16.
tiny = np.array([1e-4, 1e-5, 1e-6, 1e-7, 1e-8], dtype=np.float32)
print(f"{'value':>10s} {'as float16':>14s} {'x 1024, as float16':>22s}")
for v in tiny:
    print(f"{v:10.1e} {np.float16(v):>14.1e} "
          f"{np.float16(v * 1024):>22.1e}")

Expected output:

```
  1.0e-07        1.2e-07                 1.0e-04
  1.0e-08        0.0e+00                 1.0e-05
```

**1e-8 rounds to zero in float16, and multiplying the loss by 1024 rescues it.** Gradients are proportional to the loss, so a large scalar factor moves them into the representable range; the optimizer divides it back out before updating.

In [ ]:
keras.utils.set_random_seed(0)
m = keras.Sequential([layers.Dense(512, activation="relu"),
                      layers.Dense(512, activation="relu"),
                      layers.Dense(10, activation="softmax")])
m.compile(
    optimizer=keras.optimizers.LossScaleOptimizer(
        keras.optimizers.Adam(learning_rate=1e-3)),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
m.fit(x, y, epochs=3, batch_size=128, verbose=0)
print("with LossScaleOptimizer:", m.evaluate(xt, yt, verbose=0)[1])
print()
print("A fixed factor also works:")
print("  keras.optimizers.Adam(learning_rate=1e-3, loss_scale_factor=10)")
print("but LossScaleOptimizer adapts -- and the right value changes")
print("over the course of training.")

## float8: why it is not simply the next step

In [ ]:
print("float16 is the LAST level that 'just works'.")
print()
print("At float8 you lose too much information. Keras has a built-in")
print("implementation, but it:")
print("  - covers only Dense, EinsumDense, and Embedding")
print("  - tracks past activations to rescale each step")
print("  - overrides part of the backward pass to do the same for gradients")
print()
print("That machinery has a cost. Below roughly 5 BILLION parameters, or")
print("on anything short of an H100, the cost exceeds the benefit and you")
print("get a SLOWDOWN. float8 is rare outside foundation-model training.")

## Restore the default before continuing

In [ ]:
keras.config.set_dtype_policy("float32")
print("back to float32")

> **Note** — The dtype policy is **global**. Leaving it set affects every model built afterwards in the same process, which is a confusing afternoon if you forget.

---

## What to take away

- float16 resolves to about 1e-3; typical weight updates are 1e-6, so training in it alone fails.
- Mixed precision computes in float16 and stores variables in float32 — most of the speed, none of the instability.
- **Loss scaling** rescues gradients that would round to zero; `LossScaleOptimizer` adapts the factor.
- float8 needs 5B+ parameters and recent hardware to pay for itself.